# Projeto: Modelo de Crédito

Projeto prático com objetivo de aplicar conhecimento e habilidades de Ingestão e tratamento dos dados, até sua utilização para Análise de Dados ou modelos de Data Science

## 1 - Gerando dataset sintético em Python

In [0]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

random.seed(42) #padronizando a semente, para sempre obter os mesmo resultados no random
segmentos = ['Micro', 'Pequena', 'Média', 'Grande']
ufs = ['SP', 'RJ', 'MG', 'RS', 'PR', 'BA']
setores = ['Comércio', 'Indústria', 'Serviços', 'Agronegócio', 'Tecnologia']
setores_risco = {
    'Comércio': 0.10,
    'Indústria': 0.08,
    'Serviços': 0.05,
    'Agronegócio': 0.12,
    'Tecnologia': 0.06
    }
score = [1000, 750, 500, 250]

registros = []

for i in range(1, 2001):
    data_orig = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 600))
    segmento = random.choice(segmentos)
    setor = random.choice(setores)
    score_credito = random.choice(score)
    faturamento_mensal = 0
    valor_concedido = 0
    taxa_juros = 0
    if segmento == 'Micro':
        faturamento_mensal = round(random.uniform(5000, 32000), 2)
        valor_concedido = round(random.uniform(5000, 100000), 2)

        if score_credito >= 750:
            taxa_juros = round(random.uniform(4.0, 5.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 20, 5, 3, 2])[0]
        elif score_credito == 500:
            taxa_juros = round(random.uniform(5.0, 6.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[60, 30, 5, 3, 2])[0]
        else:
            taxa_juros = round(random.uniform(6.0, 7.0), 2)
            dias_atraso = random.choices([30, 45, 90, 120], weights=[50, 35, 10, 5])[0]

    elif segmento == 'Pequena':
        faturamento_mensal = round(random.uniform(32000, 400000), 2)
        valor_concedido = round(random.uniform(50000, 2000000), 2)

        if score_credito >= 750:
            taxa_juros = round(random.uniform(3.0, 4.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 25, 3, 1, 1])[0]
        elif score_credito == 500:
            taxa_juros = round(random.uniform(4.0, 5.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 20, 5, 4, 1])[0]
        else:
            taxa_juros = round(random.uniform(5.0, 6.0), 2)
            dias_atraso = random.choices([30, 45, 90, 120], weights=[60, 35, 4, 1])[0]

    elif segmento == 'Média':
        faturamento_mensal = round(random.uniform(400000, 25000000), 2)
        valor_concedido = round(random.uniform(500000, 100000000), 2)
        
        if score_credito >= 750:
            taxa_juros = round(random.uniform(1.5, 2.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 2, 1])[0]
        elif score_credito == 500:
            taxa_juros = round(random.uniform(2.0, 2.5), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 15, 5, 3, 2])[0]
        else:
            taxa_juros = round(random.uniform(2.5, 3.0), 2)
            dias_atraso = random.choices([30, 45, 90, 120], weights=[70, 25, 4, 1])[0]

    else:
        faturamento_mensal = round(random.uniform(25000000, 83000000), 2)
        valor_concedido = round(random.uniform(25000000, 300000000), 2)
        
        if score_credito >= 750:
            taxa_juros = round(random.uniform(1.0, 1.5), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 1, 1])[0]
        elif score_credito == 500:
            taxa_juros = round(random.uniform(1.5, 2.0), 2)
            dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 2, 1])[0]
        else:
            taxa_juros = round(random.uniform(2.0, 2.5), 2)
            dias_atraso = random.choices([30, 45, 90, 120], weights=[60, 35, 4, 1])[0]

    margem_ebitda = round(random.uniform(0.01, 0.25), 2)
    ebitda_anual = margem_ebitda * (faturamento_mensal*12)
    endividamento = min(valor_concedido / ebitda_anual, 3) # Endividamento com CAP de 3x para não quebrar o modelo com outliers
    tempo_empresa = round(random.uniform(5, 50))

    # Simulando regra de análise de crédito
    logit = (
        -3.0                        # Baseline, se os outros valores forem nulos, resulta em taxa de inadimplência de ~4,7%
        + (-1.4 if score_credito == 1000 else -0.47 if score_credito == 750 else 0.4 if score_credito == 500 else 1.4)   # Score melhor = Risco menor
        + (endividamento * 0.5)       # Quanto maior o endividamento, maior o risco
        + setores_risco[setor] * 8     # Considerando o risco do setor
        + (-0.03) * tempo_empresa    # Empresa madura = menor risco
        + np.random.normal(0, 0.5)   # Fatores macroeconômicos aleatórios que podem impactar o risco, o que acontece na realidade.
    )

    prob_indadimplencia = 1 / (1 + np.exp(-logit))  # Regressão Logística utilizando Função sigmoide para obter um resultado 0 ou 1 (inadimplente ou não inadimplente)
    inadimplente = 1 if random.random() < prob_indadimplencia else 0

    registros.append({
        'id_proposta': i,
        'id_empresa': f'EMP{i:05d}',
        'data_originacao': data_orig.strftime('%Y-%m-%d'),
        'segmento': segmento,
        'uf': random.choice(ufs),
        'setor_economico': setor,
        'valor_concedido': valor_concedido,
        'faturamento_mensal': faturamento_mensal,
        'endividamento': endividamento,
        'taxa_juros':taxa_juros,
        'tempo_empresa_anos': tempo_empresa,
        'score_credito': score_credito,
        'dias_atraso': dias_atraso,
        'inadimplente_90d': inadimplente,
        'prob_inadimplencia': prob_indadimplencia
    })

df_pd = pd.DataFrame(registros)
df_spark = spark.createDataFrame(df_pd)
df_spark.write.format('delta').mode('overwrite').option('overwriteSchema', 'True').saveAsTable('workspace.default.propostas_raw')

df_spark.show()

## 2 - Carregando a fonte de dados utilizando arquitetura Medalhão: Camada Bronze

In [0]:
df_bronze = spark.read.table('propostas_raw')
df_bronze.write.format('delta').mode('overwrite').option('overwriteSchema','True').saveAsTable('workspace.default.bronze_propostas_credito')

## 3 - Tratando os dados e gerando a Camada Silver

In [0]:
from pyspark.sql import functions as F

df_silver = df_bronze\
    .dropDuplicates(['id_proposta'])\
    .withColumn('data_originacao', F.to_date('data_originacao', 'yyy-MM-dd'))\
    .withColumn('valor_concedido', F.col("valor_concedido").cast('decimal(15,2)'))\
    .withColumn('uf', F.concat(F.lit('BR-'), F.upper(F.trim(F.col('uf')))))\
    .withColumn('mes_originacao', F.date_format('data_originacao', 'yyyy-MM'))\
    .filter(F.col('id_empresa').isNotNull()) # Se não tiver empresa, registro é inválido

df_silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'True').saveAsTable('workspace.default.silver_propostas_credito')

## 4 - Preparando os dados para consumo, Camada Gold

In [0]:
df_gold_kpis = df_silver.groupBy('segmento', 'uf')\
    .agg(
        F.count('id_proposta').alias('qtd_contratos'),
        F.sum("valor_concedido").alias('carteira_total'),
        F.avg('valor_concedido').alias('ticket_medio'),
        F.sum(F.when(F.col('dias_atraso') > 90, 1).otherwise(0)).alias('qtd_inadimplentes')
    )\
    .withColumn('taxa_inadimplencia', F.col('qtd_inadimplentes') / F.col('qtd_contratos'))

df_gold_kpis.write.format('delta').mode('overwrite').option('overwriteSchema', 'True').saveAsTable('workspace.default.gold_kpis_carteira')

In [0]:
df_gold_kpis.show()

## 5 - Construindo Vintage Analysis
Vintage Analysis é utilizada para comparar a evolução de um indicador ou valor específico durante o periodo analisado

Gerando uma nova tabela bronze com o MOB (Months on Book, mede a idade do contrato, com base na sua data de originação) mensal dos contratos, para medir a taxa de inadimplência conforme a idade dos contratos e sua variação.

In [0]:
from dateutil.relativedelta import relativedelta
from datetime import date
import numpy as np

np.random.seed(42)

contratos = spark.sql('''
    SELECT id_proposta, data_originacao, prob_inadimplencia
    FROM workspace.default.propostas_raw
''').collect()

hoje = date(2025, 8, 1)
historico = []

for contrato in contratos:
    data_orig_str = contrato['data_originacao']
    data_orig = datetime.strptime(data_orig_str, '%Y-%m-%d').date()
    mes_atual = date(data_orig.year, data_orig.month, 1)
    mob_atual = 0
    ja_inadimplente = False

    # reaproveita o logit inicial
    prob_mensal = contrato['prob_inadimplencia'] / 12

    while mes_atual <= hoje and mob_atual <= 12:
        if not ja_inadimplente:
            if np.random.random() < prob_mensal:
                ja_inadimplente = True

        historico.append({
            'id_proposta': contrato['id_proposta'],
            'mes_referencia': mes_atual.strftime('%Y-%m'),
            'mes_originacao': data_orig.strftime('%Y-%m'),
            'mob': mob_atual,
            'dias_atraso': 120 if ja_inadimplente else 0
        })
        mes_atual += relativedelta(months=1)
        mob_atual += 1

df_mob = spark.createDataFrame(historico)
df_mob.write.format('delta').mode('overwrite').option('overwriteSchema', 'True') \
    .saveAsTable('workspace.default.bronze_historico_mensal')

In [0]:
%sql
WITH flags AS (
    SELECT
        id_proposta,
        mes_originacao,
        mob,
        CASE WHEN dias_atraso > 90 THEN 1 ELSE 0 END AS inadimplente_mes,
        MAX(CASE WHEN dias_atraso > 90 THEN 1 ELSE 0 END) OVER (
            PARTITION BY id_proposta ORDER BY mob
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS ever_inadimplente
    FROM workspace.default.bronze_historico_mensal
),
vintage AS (
    SELECT
        mes_originacao,
        mob,
        COUNT(*) AS qtd_contratos,
        SUM(ever_inadimplente) AS qtd_ever_inadimplentes,
        ROUND(SUM(ever_inadimplente) / COUNT(*), 4) AS taxa_inadimplencia_acumulada
    FROM flags
    GROUP BY mes_originacao, mob
)
SELECT *,
    ROUND(
        taxa_inadimplencia_acumulada -
        LAG(taxa_inadimplencia_acumulada) OVER (PARTITION BY mes_originacao ORDER BY mob),
        4
    ) AS variacao_taxa_inadimplencia
FROM vintage
ORDER BY mes_originacao, mob

# APLICANDO ESTATISCA E MACHINE LEARNING

Tive que gerar outro dataset sintético, com informações mais próximas da realidade, visando agregar conhecimentos de regras de negócio e conhecimentos de data science.

## 1 - Preparando os dados antes de treinar o modelo

In [0]:
from pyspark.sql import functions as F

df_spark = df_spark.withColumn('log_faturamento', F.log1p(F.col('faturamento_mensal'))) #reduzindo o impacto de assimetrias no faturamento_mensal

df_encoded = pd.get_dummies(df_pd, columns=['segmento', 'setor_economico'], drop_first=True)


## 2 - Treinando o modelo

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

features = [column for column in df_encoded.columns if column not in ['id_proposta', 'id_empresa', 'data_originacao', 'uf', 'dias_atraso', 'inadimplente_90d']]

X = df_encoded[features]
y = df_encoded['inadimplente_90d']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo.fit(X_train_scaled, y_train)

## 3 - Avaliando o Modelo

In [0]:
# MLFlow
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt

mlflow.set_experiment("/Users/gui_augustonunes@outlook.com/credito_pj_pd_model")

with mlflow.start_run(run_name="regressao_logistica_v1"):

    # 1. Logar os parâmetros do modelo
    mlflow.log_param("modelo", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("features", features)  # lista de variáveis usadas
    mlflow.log_param("cap_endividamento", 3)

    # 2. Treinar (o que você já tem)
    modelo.fit(X_train_scaled, y_train)

    # 3. Calcular e logar métricas
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ks = max(tpr - fpr)

    mlflow.log_metric("auc", auc)
    mlflow.log_metric("ks", ks)
    mlflow.log_metric("taxa_inadimplencia_base", y.mean())

    # 4. Logar um artefato (ex: gráfico da curva ROC)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0,1],[0,1], linestyle="--", color="gray")
    plt.xlabel("Falso Positivo"); plt.ylabel("Verdadeiro Positivo")
    plt.legend()
    plt.savefig("roc_curve.png")
    mlflow.log_artifact("roc_curve.png")

    # 5. Logar o próprio modelo treinado (o mais importante)
    mlflow.sklearn.log_model(modelo, "modelo_pd_credito_pj")

In [0]:
regressao_logistica_v1 = 'runs:/<run_id>/model'  # Replace <run_id> with the actual run IDmlflow.register_model(regressao_logistica_v1)


In [0]:
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, roc_curve)

y_pred = modelo.predict(X_test_scaled)
y_proba = modelo.predict_proba(X_test_scaled)[:, 1] # Probabilidade da classe 1

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

auc = roc_auc_score(y_test, y_proba)
print(f'AUC: {auc:.4f}')

# Utilizando o KS
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
ks = max(tpr-fpr)
print(f'KS: {ks:.4f}')

## 4 - Interpretando o peso de cada coeficiente

In [0]:
coeficientes = pd.DataFrame({
    'variavel': features,
    'coeficiente': modelo.coef_[0]
}).sort_values('coeficiente', ascending=False)
print(coeficientes)

O modelo aprendeu que o coeficiente de endividamento é o indicador de maior peso no risco de inadimplência

## 5 - Simulando monitoramento de performane do modelo com o passar do tempo

Simulando 6 meses de propostas, visando gerar um histórico para monitoramento do modelo.

In [0]:
import numpy as np
import pandas as pd

np.random.seed(100)  # seed diferente do treino, simula "novos dados"

def gerar_lote_mensal(mes_referencia, n=150, fator_drift=1.0):
    """
    fator_drift > 1 simula deterioração do perfil de risco ao longo do tempo
    (ex: mudança no mix de clientes ou piora macroeconômica)
    """
    registros = []
    for i in range(n):
        segmento = np.random.choice(segmentos)
        setor = np.random.choice(setores)
        score_credito = np.random.choice(score)
        faturamento_mensal = 0
        valor_concedido = 0
        taxa_juros = 0
        if segmento == 'Micro':
            faturamento_mensal = round(random.uniform(5000, 32000), 2)
            valor_concedido = round(random.uniform(5000, 100000), 2)

            if score_credito >= 750:
                taxa_juros = round(random.uniform(4.0, 5.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 20, 5, 3, 2])[0]
            elif score_credito == 500:
                taxa_juros = round(random.uniform(5.0, 6.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[60, 30, 5, 3, 2])[0]
            else:
                taxa_juros = round(random.uniform(6.0, 7.0), 2)
                dias_atraso = random.choices([30, 45, 90, 120], weights=[50, 35, 10, 5])[0]

        elif segmento == 'Pequena':
            faturamento_mensal = round(random.uniform(32000, 400000), 2)
            valor_concedido = round(random.uniform(50000, 2000000), 2)

            if score_credito >= 750:
                taxa_juros = round(random.uniform(3.0, 4.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 25, 3, 1, 1])[0]
            elif score_credito == 500:
                taxa_juros = round(random.uniform(4.0, 5.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 20, 5, 4, 1])[0]
            else:
                taxa_juros = round(random.uniform(5.0, 6.0), 2)
                dias_atraso = random.choices([30, 45, 90, 120], weights=[60, 35, 4, 1])[0]

        elif segmento == 'Média':
            faturamento_mensal = round(random.uniform(400000, 25000000), 2)
            valor_concedido = round(random.uniform(500000, 100000000), 2)
            
            if score_credito >= 750:
                taxa_juros = round(random.uniform(1.5, 2.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 2, 1])[0]
            elif score_credito == 500:
                taxa_juros = round(random.uniform(2.0, 2.5), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[70, 15, 5, 3, 2])[0]
            else:
                taxa_juros = round(random.uniform(2.5, 3.0), 2)
                dias_atraso = random.choices([30, 45, 90, 120], weights=[70, 25, 4, 1])[0]

        else:
            faturamento_mensal = round(random.uniform(25000000, 83000000), 2)
            valor_concedido = round(random.uniform(25000000, 300000000), 2)
            
            if score_credito >= 750:
                taxa_juros = round(random.uniform(1.0, 1.5), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 1, 1])[0]
            elif score_credito == 500:
                taxa_juros = round(random.uniform(1.5, 2.0), 2)
                dias_atraso = random.choices([0, 30, 45, 90, 120], weights=[80, 15, 3, 2, 1])[0]
            else:
                taxa_juros = round(random.uniform(2.0, 2.5), 2)
                dias_atraso = random.choices([30, 45, 90, 120], weights=[60, 35, 4, 1])[0]

        margem_ebitda = round(random.uniform(0.01, 0.25), 2)
        ebitda_anual = margem_ebitda * (faturamento_mensal*12)
        endividamento = min(valor_concedido / ebitda_anual, 3) # Endividamento com CAP de 3x para não quebrar o modelo com outliers
        tempo_empresa = round(random.uniform(5, 50))

        logit = (
            -3.0
            + (-1.4 if score_credito == 1000 else -0.47 if score_credito == 750 else 0.4 if score_credito == 500 else 1.4)
            + (endividamento * 0.5) * fator_drift   # <- drift aplicado aqui
            + setores_risco[setor] * 8
            + (-0.03) * tempo_empresa
            + np.random.normal(0, 0.5)
        )
        prob = 1 / (1 + np.exp(-logit))
        inadimplente_real = 1 if np.random.random() < prob else 0

        registros.append({
            'id_proposta': i,
            'id_empresa': f'EMP{i:05d}',
            'data_originacao': data_orig.strftime('%Y-%m-%d'),
            'segmento': segmento,
            'uf': random.choice(ufs),
            'setor_economico': setor,
            'valor_concedido': valor_concedido,
            'faturamento_mensal': faturamento_mensal,
            'endividamento': endividamento,
            'taxa_juros':taxa_juros,
            'tempo_empresa_anos': tempo_empresa,
            'score_credit': score_credito,
            'dias_atraso': dias_atraso,
            'inadimplente_real': inadimplente_real,
            'mes_referencia': mes_referencia
        })
    return pd.DataFrame(registros)

# Simula 6 meses, com deterioração gradual a partir do mês 4 (simulando uma crise ou fator macroeconômico)
meses = ["2025-09", "2025-10", "2025-11", "2025-12", "2026-01", "2026-02"]
fatores_drift = [1.0, 1.0, 1.0, 1.3, 1.5, 1.6]  # piora a partir de dezembro

lotes = [gerar_lote_mensal(mes, fator_drift=f) for mes, f in zip(meses, fatores_drift)]
df_producao = pd.concat(lotes, ignore_index=True)

## 6 - utilizando o modelo já treinado para calcular métricas mensais

In [0]:
# aplicando o MESMO encoding do treino
df_producao_encoded = pd.get_dummies(df_producao, columns=['segmento', 'setor_economico'], drop_first=True)

# Renomear score_credit para score_credito para corresponder ao treino
df_producao_encoded = df_producao_encoded.rename(columns={'score_credit': 'score_credito'})

# aplicando o MESMO scaler do treino
X_producao = df_producao_encoded[features]  # mesmas colunas do treino, mesma ordem
X_producao_scaled = scaler.transform(X_producao)  # reutilizando o scaler já ajustado
df_producao["score_modelo"] = modelo.predict_proba(X_producao_scaled)[:, 1]

from sklearn.metrics import roc_auc_score

resultados_mensais = []
for mes in meses:
    subset = df_producao[df_producao["mes_referencia"] == mes]
    auc_mes = roc_auc_score(subset["inadimplente_real"], subset["score_modelo"])
    taxa_real = subset["inadimplente_real"].mean()
    score_medio = subset["score_modelo"].mean()

    fpr, tpr, _ = roc_curve(subset["inadimplente_real"], subset["score_modelo"])
    ks_mes = max(tpr - fpr)

    resultados_mensais.append({
        "mes_referencia": mes,
        "qtd_propostas": len(subset),
        "taxa_inadimplencia_real": round(taxa_real, 4),
        "score_medio_modelo": round(score_medio, 4),
        "auc": round(auc_mes, 4),
        "ks": round(ks_mes, 4)
    })

df_monitoramento = pd.DataFrame(resultados_mensais)
print(df_monitoramento)

In [0]:
df_monitoramento_spark = spark.createDataFrame(df_monitoramento)
df_monitoramento_spark.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_monitoramento_modelo_pd")